# 02 — Data Transformation and Preparation

### Notebook roadmap
1. Setup and loading
2. Initial cleaning (non-learned, row-level operations)
3. Train / test split
4. Missing value strategy
5. Outlier strategy
6. Feature engineering
7. Categorical encoding
8. Class imbalance strategy
9. Scaling
10. Assembling the full preprocessing pipeline
11. Validating the pipeline (leakage check, sanity checks)
12. Saving the artefacts
13. Summary

## 1. Setup and loading

### 1.1 Imports and configuration


In [1]:
import sys 
from pathlib import Path 
ROOT =Path .cwd ().parent 
sys .path .append (str (ROOT ))
import numpy as np 
import pandas as pd 
import matplotlib .pyplot as plt 
import seaborn as sns 
from src .data .load_data import load_data 


/home/neo/.config/matplotlib is not a writable directory
Matplotlib created a temporary cache directory at /tmp/matplotlib-n3l1ppy0 because there was an issue with the default path (/home/neo/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


### 1.2 Load the raw data

In [2]:
df =load_data ('raw')
print (df .info ())

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Student_ID                      2000 non-null   str    
 1   Age                             2000 non-null   int64  
 2   Gender                          2000 non-null   str    
 3   Year_of_Study                   2000 non-null   int64  
 4   Sleep_Hours                     1938 non-null   float64
 5   Study_Hours_Per_Week            2000 non-null   float64
 6   GPA                             1962 non-null   float64
 7   Social_Media_Hours              1942 non-null   float64
 8   Screen_Time_Hours               2000 non-null   float64
 9   Physical_Activity               2000 non-null   str    
 10  Financial_Stress                1909 non-null   float64
 11  Family_Support                  1946 non-null   float64
 12  Attendance_Rate                 1953 non-null

### 1.3 Recall the decisions from the EDA

In [3]:
identifier ="Student_ID"
TARGET_COL ="Mental_Health_Risk"
NUMERICAL_COLUMNS =df .select_dtypes (include =['int64',
'float64']).columns .tolist ()
CATEGORICAL_COLUMNS =df .select_dtypes (include =['str']).columns .tolist ()
CATEGORICAL_COLUMNS .remove (identifier )
CATEGORICAL_COLUMNS .remove (TARGET_COL )
print ("numerical: ")
print (NUMERICAL_COLUMNS )
print ("\n\n")
print (CATEGORICAL_COLUMNS )
ID_COLS =["Student_ID"]
NUM_CONT_COLS =["Age","Sleep_Hours","Study_Hours_Per_Week","GPA",
"Social_Media_Hours","Screen_Time_Hours","Attendance_Rate"]
NUM_DISC_COLS =["Year_of_Study","Financial_Stress","Family_Support"]
ORDINAL_COLS =["Physical_Activity","Diet_Quality"]
NOMINAL_COLS =["Gender"]
BINARY_COLS =["Part_Time_Job","Previous_Mental_Health_History"]
ORDINAL_ORDERS ={
"Physical_Activity":["Low","Moderate","High"],
"Diet_Quality":["Poor","Average","Good"],
}

numerical: 
['Age', 'Year_of_Study', 'Sleep_Hours', 'Study_Hours_Per_Week', 'GPA', 'Social_Media_Hours', 'Screen_Time_Hours', 'Financial_Stress', 'Family_Support', 'Attendance_Rate']



['Gender', 'Physical_Activity', 'Part_Time_Job', 'Diet_Quality', 'Previous_Mental_Health_History']


## 2. Initial cleaning

### 2.1 Remove the identifier

In [4]:
df =df .drop (columns =[identifier ])

### 2.2 Handle duplicates
skipped due to nonexisting dups

### 2.3 Standardise categorical text

In [5]:
for column in CATEGORICAL_COLUMNS :
    df [column ]=df [column ].str .capitalize ()
    df [column ]=df [column ].str .strip ()

### 2.4 Handle impossible values


In [6]:
df ['Social_Media_Hours']=np .where (df ['Social_Media_Hours']>df ['Screen_Time_Hours'],np .nan ,df ['Social_Media_Hours'])

### 2.5 Check the target


In [7]:
target =df [TARGET_COL ]
features =df .drop (columns =[TARGET_COL ])

## 3. Train / test split

### 3.1 Perform the split

### 3.2 Verify the split

### 3.3 validation set


In [8]:
from sklearn .model_selection import train_test_split 
x_train ,x_test ,y_train ,y_test =train_test_split (features ,target ,stratify =target ,random_state =42 )
print (f"train features: {x_train .shape } | train target: {y_train .shape }")
print (f"test features: {x_test .shape } | test target: {y_test .shape }")


train features: (1500, 15) | train target: (1500,)
test features: (500, 15) | test target: (500,)


## 4. Missing value strategy


In [9]:
from sklearn .impute import SimpleImputer 
numerical_imputer =SimpleImputer (strategy ='median')
categorical_imputer =SimpleImputer (strategy ='most_frequent')

### 4.2  imputation


In [10]:
from sklearn .compose import ColumnTransformer 
transformer =ColumnTransformer (
transformers =[
('num',numerical_imputer ,NUMERICAL_COLUMNS ),
('cat',categorical_imputer ,CATEGORICAL_COLUMNS )
],
remainder ='passthrough',
verbose_feature_names_out =False 
).set_output (transform ='pandas')
x_train =transformer .fit_transform (x_train )
x_test =transformer .transform (x_test )

## 5. Outlier strategy

### 5.1 Outlier detection using the 1.5 × IQR rule
We evaluate extreme values across continuous numerical variables (`NUM_CONT_COLS`) using the standard Tukey Interquartile Range (IQR) rule:

$$\text{Lower Bound} = Q_1 - 1.5 \times \text{IQR}, \quad \text{Upper Bound} = Q_3 + 1.5 \times \text{IQR}$$

> **Fix Applied:**
> 1. **Threshold adjustment:** Previously, an excessively wide multiplier of $4 \times \text{IQR}$ was used, resulting in zero outliers being flagged (e.g., Sleep Hours lower bound was 0.0, GPA lower bound was 0.45). Switching to the standard $1.5 \times \text{IQR}$ correctly identifies true distribution tails.
> 2. **Loop condition correction:** In the feature inspection loop, the upper condition was checking `(x > LOWER_BOUND)` instead of `(x > UPPER_BOUND)`, which caused almost all samples to be erroneously selected. It is now corrected to `(x > UPPER_BOUND)`.


In [11]:
Q1 =x_train [NUM_CONT_COLS ].quantile (0.25 )
Q3 =x_train [NUM_CONT_COLS ].quantile (0.75 )
IQR =Q3 -Q1 
LOWER_BOUND =Q1 -1.5 *IQR 
UPPER_BOUND =Q3 +1.5 *IQR 
bounds_summary =pd .DataFrame ({
'Min':x_train [NUM_CONT_COLS ].min ().round (2 ),
'Q1 (25%)':Q1 .round (2 ),
'Median':x_train [NUM_CONT_COLS ].median ().round (2 ),
'Q3 (75%)':Q3 .round (2 ),
'Max':x_train [NUM_CONT_COLS ].max ().round (2 ),
'IQR':IQR .round (2 ),
'Lower Bound (1.5*IQR)':LOWER_BOUND .round (2 ),
'Upper Bound (1.5*IQR)':UPPER_BOUND .round (2 )
})
bounds_summary 


In [12]:
outlier_mask =(
(x_train [NUM_CONT_COLS ]<LOWER_BOUND )|
(x_train [NUM_CONT_COLS ]>UPPER_BOUND )
)


In [13]:
outlier_summary =pd .DataFrame ({
'outlier_count':outlier_mask .sum (),
'percentage (%)':(outlier_mask .mean ()*100 ).round (2 ),
'lower_bound':LOWER_BOUND .round (2 ),
'upper_bound':UPPER_BOUND .round (2 ),
'min_observed':x_train [NUM_CONT_COLS ].min ().round (2 ),
'max_observed':x_train [NUM_CONT_COLS ].max ().round (2 )
})
print (f"Total outlier points across continuous features: {outlier_mask .sum ().sum ()}")
print (f"Total rows with at least one outlier: {outlier_mask .any (axis =1 ).sum ()} ({outlier_mask .any (axis =1 ).mean ()*100 :.2f}%)")
outlier_summary .sort_values ('outlier_count',ascending =False )


Total outlier points across continuous features: 56
Total rows with at least one outlier: 51 (3.40%)


In [ ]:
for column in NUM_CONT_COLS :
    mask =(
    (x_train [column ]<LOWER_BOUND [column ])|
    (x_train [column ]>UPPER_BOUND [column ])
    )
    n_outliers =mask .sum ()
    if n_outliers >0 :
        print (f"======= {column } ({n_outliers } outliers outside [{LOWER_BOUND [column ]:.2f}, {UPPER_BOUND [column ]:.2f}]) =======")
        display (x_train .loc [mask ,[column ]].head (5 ))


,Age
1615,28.0
1957,28.0


,Sleep_Hours
650,9.8
749,3.6
1580,10.0
688,10.0
66,10.0


,Study_Hours_Per_Week
1042,2.7
453,3.2
937,37.5
716,38.3
866,2.4


,GPA
937,3.64
1320,3.51
1299,3.57
448,1.50
1923,3.51


,Social_Media_Hours
1087,10.0
453,10.0
768,9.0
485,2.9
470,9.0


,Screen_Time_Hours
1169,3.3
1466,3.8
1754,14.0
485,3.8
1487,13.6


,Attendance_Rate
523,62.5
36,61.1
277,62.4
1721,61.9
1407,61.8


======= Age (2 outliers outside [15.50, 27.50]) =======
======= Sleep_Hours (8 outliers outside [3.75, 9.75]) =======
======= Study_Hours_Per_Week (7 outliers outside [4.00, 36.00]) =======
======= GPA (6 outliers outside [1.63, 3.50]) =======
======= Social_Media_Hours (16 outliers outside [3.20, 8.80]) =======
======= Screen_Time_Hours (7 outliers outside [3.95, 13.15]) =======
======= Attendance_Rate (10 outliers outside [65.75, 104.55]) =======


### 5.2 Outlier Treatment: Capping (Winsorization) vs. Row Deletion


In [ ]:
for col in NUM_CONT_COLS :
    x_train [col ]=x_train [col ].clip (lower =LOWER_BOUND [col ],upper =UPPER_BOUND [col ])
    x_test [col ]=x_test [col ].clip (lower =LOWER_BOUND [col ],upper =UPPER_BOUND [col ])
remaining_outliers =(
(x_train [NUM_CONT_COLS ]<LOWER_BOUND )|
(x_train [NUM_CONT_COLS ]>UPPER_BOUND )
).sum ()
print ("Verification: Outliers remaining in x_train after capping:")
print (remaining_outliers )
assert remaining_outliers .sum ()==0 ,"Outliers still detected after capping!"
print ("\nAll continuous numerical outliers successfully capped to [Lower Bound, Upper Bound].")


Verification: Outliers remaining in x_train after capping:
Age                     0
Sleep_Hours             0
Study_Hours_Per_Week    0
GPA                     0
Social_Media_Hours      0
Screen_Time_Hours       0
Attendance_Rate         0
dtype: int64

All continuous numerical outliers successfully capped to [Lower Bound, Upper Bound].


## 6. Feature engineering

### 6.1 Design domain candidate features
We construct intuitive candidate interaction features capturing student lifestyle, academic workload, and mental health risk drivers:
- **Sleep Deficit:** `max(0, 7.5 - Sleep_Hours)` — deficit below the recommended 7.5 hours of sleep.
- **Study-to-Sleep Ratio:** `Study_Hours_Per_Week / (Sleep_Hours * 7 + 1e-5)` — academic study load relative to weekly sleep rest.
- **Social Media Share:** `Social_Media_Hours / (Screen_Time_Hours + 1e-5)` — share of total screen time spent on social media.
- **Non-Social Screen Time:** `max(0, Screen_Time_Hours - Social_Media_Hours)` — screen time spent on productivity, studying, or other non-social activities.
- **Stress-to-Support Ratio:** `Financial_Stress / (Family_Support + 1e-5)` — ratio of financial burden against family emotional safety net.
- **Academic Effort Index:** `GPA * (Study_Hours_Per_Week / 10.0)` — composite measure of performance relative to weekly study effort.


In [ ]:
def engineer_candidate_features (df_in :pd .DataFrame )->pd .DataFrame :
    df_feat =df_in .copy ()
    df_feat ['Sleep_Deficit']=np .maximum (0.0 ,7.5 -df_feat ['Sleep_Hours'])
    df_feat ['Study_to_Sleep_Ratio']=df_feat ['Study_Hours_Per_Week']/(df_feat ['Sleep_Hours']*7.0 +1e-5 )
    df_feat ['Social_Media_Share']=df_feat ['Social_Media_Hours']/(df_feat ['Screen_Time_Hours']+1e-5 )
    df_feat ['Non_Social_Screen_Time']=np .maximum (0.0 ,df_feat ['Screen_Time_Hours']-df_feat ['Social_Media_Hours'])
    df_feat ['Stress_to_Support_Ratio']=df_feat ['Financial_Stress']/(df_feat ['Family_Support']+1e-5 )
    df_feat ['Academic_Effort_Index']=df_feat ['GPA']*(df_feat ['Study_Hours_Per_Week']/10.0 )
    return df_feat 
x_train_candidates =engineer_candidate_features (x_train )
x_test_candidates =engineer_candidate_features (x_test )
print (f"Candidate feature generation complete: {x_train .shape [1 ]} original -> {x_train_candidates .shape [1 ]} total features")
display (x_train_candidates [[
'Sleep_Deficit',
'Study_to_Sleep_Ratio',
'Social_Media_Share',
'Non_Social_Screen_Time',
'Stress_to_Support_Ratio',
'Academic_Effort_Index'
]].head (5 ))


,Sleep_Deficit,Study_to_Sleep_Ratio,Social_Media_Share,Non_Social_Screen_Time,Stress_to_Support_Ratio,Academic_Effort_Index
419,0.5,0.426531,0.673684,3.1,0.736841,5.9356
1144,0.0,0.327306,0.635593,4.3,0.873683,5.7196
1806,0.5,0.606122,0.580644,2.6,1.013157,8.8209
746,1.7,0.701970,0.874998,0.7,1.370966,8.6925
661,1.0,0.349450,0.724489,2.7,3.285699,4.3884


Candidate feature generation complete: 15 original -> 21 total features


### 6.2 Simplified Feature Selection: Variance & Correlation Filters

To avoid unnecessary complexity and retain only informative, non-redundant features, we apply a clear 3-step filtering pipeline:
1. **Low Variance Filter:** Drop features with near-zero variance ($\sigma^2 < 0.02$) that lack discriminatory spread across individuals.
2. **Multicollinearity / Redundancy Filter:** Detect feature pairs with high pairwise correlation ($|r| \ge 0.75$). For each redundant pair, compare correlation with the target `Mental_Health_Risk`, keep the more predictive one, and drop the redundant feature.
3. **Low Target Correlation Filter:** Drop features with negligible correlation ($|r| < 0.05$) to the target risk level.


In [17]:
target_mapping ={'Low':0 ,'Medium':1 ,'High':2 }
y_train_num =y_train .map (target_mapping )
numeric_candidates =x_train_candidates .select_dtypes (include =[np .number ]).columns .tolist ()
VARIANCE_THRESHOLD =0.02 
variances =x_train_candidates [numeric_candidates ].var ()
low_var_features =variances [variances <VARIANCE_THRESHOLD ].index .tolist ()
print (f"=== Filter 1: Low Variance (< {VARIANCE_THRESHOLD }) ===")
if low_var_features :
    for f in low_var_features :
        print (f"  [DROP] '{f }' (variance = {variances [f ]:.4f})")
else :
    print ("  None detected.")
candidates_pass_var =[c for c in numeric_candidates if c not in low_var_features ]
CORR_THRESHOLD =0.75 
target_corrs =x_train_candidates [candidates_pass_var ].apply (lambda col :col .corr (y_train_num )).abs ()
pairwise_corr =x_train_candidates [candidates_pass_var ].corr ().abs ()
redundant_drops =set ()
print (f"\n=== Filter 2: Multicollinearity / Redundancy (|r| >= {CORR_THRESHOLD }) ===")
for i in range (len (candidates_pass_var )):
    for j in range (i +1 ,len (candidates_pass_var )):
        f1 =candidates_pass_var [i ]
        f2 =candidates_pass_var [j ]
        r_val =pairwise_corr .loc [f1 ,f2 ]
        if r_val >=CORR_THRESHOLD :
            target_r1 =target_corrs [f1 ]
            target_r2 =target_corrs [f2 ]
            drop_f =f2 if target_r1 >=target_r2 else f1 
            keep_f =f1 if target_r1 >=target_r2 else f2 
            redundant_drops .add (drop_f )
            print (f"  Pair ({f1 }, {f2 }): |r| = {r_val :.3f} -> Keep '{keep_f }' (target_r={target_corrs [keep_f ]:.3f}), Drop '{drop_f }' (target_r={target_corrs [drop_f ]:.3f})")
candidates_pass_corr =[c for c in candidates_pass_var if c not in redundant_drops ]
TARGET_CORR_THRESHOLD =0.05 
rem_target_corrs =target_corrs [candidates_pass_corr ]
low_target_corr_features =rem_target_corrs [rem_target_corrs <TARGET_CORR_THRESHOLD ].index .tolist ()
print (f"\n=== Filter 3: Low Target Correlation (< {TARGET_CORR_THRESHOLD }) ===")
if low_target_corr_features :
    for f in low_target_corr_features :
        print (f"  [DROP] '{f }' (|r_target| = {rem_target_corrs [f ]:.4f})")
else :
    print ("  None detected.")
selected_numeric_features =[c for c in candidates_pass_corr if c not in low_target_corr_features ]
print (f"\nSelection Result: {len (numeric_candidates )} candidate numerical features -> {len (selected_numeric_features )} retained.")


=== Filter 1: Low Variance (< 0.02) ===
  [DROP] 'Social_Media_Share' (variance = 0.0147)

=== Filter 2: Multicollinearity / Redundancy (|r| >= 0.75) ===
  Pair (Sleep_Hours, Sleep_Deficit): |r| = 0.949 -> Keep 'Sleep_Hours' (target_r=0.538), Drop 'Sleep_Deficit' (target_r=0.516)
  Pair (Study_Hours_Per_Week, Study_to_Sleep_Ratio): |r| = 0.842 -> Keep 'Study_Hours_Per_Week' (target_r=0.249), Drop 'Study_to_Sleep_Ratio' (target_r=0.056)
  Pair (Study_Hours_Per_Week, Academic_Effort_Index): |r| = 0.944 -> Keep 'Academic_Effort_Index' (target_r=0.317), Drop 'Study_Hours_Per_Week' (target_r=0.249)
  Pair (Study_to_Sleep_Ratio, Academic_Effort_Index): |r| = 0.750 -> Keep 'Academic_Effort_Index' (target_r=0.317), Drop 'Study_to_Sleep_Ratio' (target_r=0.056)

=== Filter 3: Low Target Correlation (< 0.05) ===
  [DROP] 'Age' (|r_target| = 0.0200)

Selection Result: 16 candidate numerical features -> 11 retained.


### 6.3 Diagnostic Visualizations and Selected Feature Assembly


In [18]:
fig ,axes =plt .subplots (1 ,2 ,figsize =(16 ,6 ))
all_target_corrs =x_train_candidates [numeric_candidates ].apply (lambda col :col .corr (y_train_num )).sort_values (key =abs ,ascending =True )
colors =['#2ecc71'if col in selected_numeric_features else '#e74c3c'for col in all_target_corrs .index ]
axes [0 ].barh (all_target_corrs .index ,all_target_corrs .values ,color =colors )
axes [0 ].axvline (0 ,color ='black',linewidth =0.8 ,linestyle ='--')
axes [0 ].axvline (TARGET_CORR_THRESHOLD ,color ='gray',linestyle =':',label =f'Target Threshold (±{TARGET_CORR_THRESHOLD })')
axes [0 ].axvline (-TARGET_CORR_THRESHOLD ,color ='gray',linestyle =':')
axes [0 ].set_title ('Feature Correlation with Target Risk (Green=Selected, Red=Dropped)')
axes [0 ].set_xlabel ('Pearson Correlation with Mental_Health_Risk')
axes [0 ].legend (loc ='lower right')
sns .heatmap (
x_train_candidates [selected_numeric_features ].corr (),
annot =True ,
fmt ='.2f',
cmap ='coolwarm',
center =0 ,
ax =axes [1 ],
cbar_kws ={'label':'Correlation'}
)
axes [1 ].set_title ('Pairwise Correlation Matrix of Selected Numerical Features')
plt .tight_layout ()
plt .show ()
categorical_features_list =[c for c in x_train_candidates .columns if c not in numeric_candidates and c in x_train .columns ]
final_features =selected_numeric_features +categorical_features_list 
x_train =x_train_candidates [final_features ].copy ()
x_test =x_test_candidates [final_features ].copy ()
print (f"Final transformed feature matrices: x_train: {x_train .shape }, x_test: {x_test .shape }")
print ("\nRetained Features:")
for i ,feat in enumerate (final_features ,1 ):
    is_new =" [NEW Engineered Feature]"if feat in ['Stress_to_Support_Ratio','Academic_Effort_Index','Non_Social_Screen_Time']else ""
    print (f" {i :2d}. {feat :28s}{is_new }")


Final transformed feature matrices: x_train: (1500, 16), x_test: (500, 16)

Retained Features:
  1. Year_of_Study               
  2. Sleep_Hours                 
  3. GPA                         
  4. Social_Media_Hours          
  5. Screen_Time_Hours           
  6. Financial_Stress            
  7. Family_Support              
  8. Attendance_Rate             
  9. Non_Social_Screen_Time       [NEW Engineered Feature]
 10. Stress_to_Support_Ratio      [NEW Engineered Feature]
 11. Academic_Effort_Index        [NEW Engineered Feature]
 12. Gender                      
 13. Physical_Activity           
 14. Part_Time_Job               
 15. Diet_Quality                
 16. Previous_Mental_Health_History


## 7. Categorical encoding

### 7.1 Choose an encoding per variable type
- **Ordinal categories (`Physical_Activity`, `Diet_Quality`):** Map to ordered integers according to their natural domain rank (`ORDINAL_ORDERS`).
- **Nominal categories (`Gender`):** One-hot encode with `drop='first'` to prevent dummy variable collinearity.
- **Binary categories (`Part_Time_Job`, `Previous_Mental_Health_History`):** Map directly to 0/1 integers.


In [19]:
from sklearn .preprocessing import OneHotEncoder ,OrdinalEncoder 
ordinal_enc =OrdinalEncoder (
categories =[
ORDINAL_ORDERS ['Physical_Activity'],
ORDINAL_ORDERS ['Diet_Quality']
],
handle_unknown ='use_encoded_value',
unknown_value =-1 
)
x_train [ORDINAL_COLS ]=ordinal_enc .fit_transform (x_train [ORDINAL_COLS ])
x_test [ORDINAL_COLS ]=ordinal_enc .transform (x_test [ORDINAL_COLS ])
for col in BINARY_COLS :
    x_train [col ]=(x_train [col ].astype (str ).str .lower ()=='yes').astype (int )
    x_test [col ]=(x_test [col ].astype (str ).str .lower ()=='yes').astype (int )
nominal_enc =OneHotEncoder (drop ='first',sparse_output =False ,handle_unknown ='ignore')
gender_train =pd .DataFrame (
nominal_enc .fit_transform (x_train [NOMINAL_COLS ]),
columns =nominal_enc .get_feature_names_out (NOMINAL_COLS ),
index =x_train .index 
)
gender_test =pd .DataFrame (
nominal_enc .transform (x_test [NOMINAL_COLS ]),
columns =nominal_enc .get_feature_names_out (NOMINAL_COLS ),
index =x_test .index 
)
x_train =pd .concat ([x_train .drop (columns =NOMINAL_COLS ),gender_train ],axis =1 ).astype (float )
x_test =pd .concat ([x_test .drop (columns =NOMINAL_COLS ),gender_test ],axis =1 ).astype (float )
print (f"Categorical encoding complete! Training features shape: {x_train .shape }, Test features shape: {x_test .shape }")
display (x_train .head (5 ))


,Year_of_Study,Sleep_Hours,GPA,Social_Media_Hours,Screen_Time_Hours,Financial_Stress,Family_Support,Attendance_Rate,Non_Social_Screen_Time,Stress_to_Support_Ratio,Academic_Effort_Index,Physical_Activity,Part_Time_Job,Diet_Quality,Previous_Mental_Health_History,Gender_Male,Gender_Non-binary
419,3.0,7.0,2.84,6.4,9.5,4.2,5.7,83.4,3.1,0.736841,5.9356,0.0,1.0,2.0,0.0,0.0,0.0
1144,3.0,7.9,3.16,7.5,11.8,8.3,9.5,94.3,4.3,0.873683,5.7196,0.0,1.0,0.0,0.0,0.0,0.0
1806,5.0,7.0,2.97,3.6,6.2,7.7,7.6,89.5,2.6,1.013157,8.8209,2.0,0.0,0.0,0.0,0.0,0.0
746,4.0,5.8,3.05,4.9,5.6,8.5,6.2,78.1,0.7,1.370966,8.6925,0.0,0.0,1.0,0.0,0.0,1.0
661,1.0,6.5,2.76,7.1,9.8,6.9,2.1,87.3,2.7,3.285699,4.3884,0.0,0.0,1.0,1.0,0.0,0.0


Categorical encoding complete! Training features shape: (1500, 17), Test features shape: (500, 17)


### 7.5 Encode the target
Encode target risk levels to consistent integers: `Low=0, Medium=1, High=2`.


In [20]:
TARGET_MAPPING ={'Low':0 ,'Medium':1 ,'High':2 }
TARGET_INVERSE_MAPPING ={0 :'Low',1 :'Medium',2 :'High'}
y_train_encoded =y_train .map (TARGET_MAPPING ).astype (int )
y_test_encoded =y_test .map (TARGET_MAPPING ).astype (int )
print ("Target encoding complete:")
print ("Training set target proportions:")
print (y_train_encoded .value_counts (normalize =True ).round (3 ))
print ("\nTest set target proportions:")
print (y_test_encoded .value_counts (normalize =True ).round (3 ))


Target encoding complete:
Training set target proportions:
Mental_Health_Risk
2    0.34
0    0.33
1    0.33
Name: proportion, dtype: float64

Test set target proportions:
Mental_Health_Risk
2    0.34
1    0.33
0    0.33
Name: proportion, dtype: float64


## 8. Class imbalance strategy
Evaluate training set target proportions to determine if resampling (e.g. SMOTE or class weighting) is justified.


In [21]:
class_counts =y_train_encoded .value_counts ().sort_index ()
class_pct =(y_train_encoded .value_counts (normalize =True ).sort_index ()*100 ).round (2 )
imbalance_df =pd .DataFrame ({
'Class':[TARGET_INVERSE_MAPPING [i ]for i in class_counts .index ],
'Count':class_counts .values ,
'Percentage (%)':class_pct .values 
})
display (imbalance_df )
print ("Decision: Target classes are naturally balanced (~33% Low, ~33% Medium, ~34% High).")
print ("Resampling is unnecessary and would introduce artificial noise without benefit.")


,Class,Count,Percentage (%)
0,Low,495,33.0
1,Medium,495,33.0
2,High,510,34.0


Decision: Target classes are naturally balanced (~33% Low, ~33% Medium, ~34% High).
Resampling is unnecessary and would introduce artificial noise without benefit.


## 9. Scaling
Apply `StandardScaler` to ensure all numerical features have zero mean and unit variance, which is required by distance and gradient-based models.


In [22]:
from sklearn .preprocessing import StandardScaler 
scaler =StandardScaler ()
x_train_scaled =pd .DataFrame (
scaler .fit_transform (x_train ),
columns =x_train .columns ,
index =x_train .index 
)
x_test_scaled =pd .DataFrame (
scaler .transform (x_test ),
columns =x_test .columns ,
index =x_test .index 
)
print (f"Scaling complete. Train mean: {x_train_scaled .mean ().mean ():.4f}, Train std: {x_train_scaled .std ().mean ():.4f}")
display (x_train_scaled .head (5 ))


,Year_of_Study,Sleep_Hours,GPA,Social_Media_Hours,Screen_Time_Hours,Financial_Stress,Family_Support,Attendance_Rate,Non_Social_Screen_Time,Stress_to_Support_Ratio,Academic_Effort_Index,Physical_Activity,Part_Time_Job,Diet_Quality,Previous_Mental_Health_History,Gender_Male,Gender_Non-binary
419,0.284557,0.237872,0.801643,0.337211,0.562352,-0.950293,-0.457459,-0.237242,0.455558,-0.550216,0.369507,-1.202264,1.352845,1.446624,-0.635999,-0.935414,-0.221249
1144,0.284557,1.059168,1.740817,1.340454,1.953664,1.244256,1.637101,1.286286,1.455806,-0.273853,0.258137,-1.202264,1.352845,-1.293194,-0.635999,-0.935414,-0.221249
1806,1.800843,0.237872,1.183183,-2.216498,-1.433878,0.923102,0.589821,0.615375,0.038787,0.007824,1.857165,1.567929,-0.739183,-1.293194,-0.635999,-0.935414,-0.221249
746,1.042700,-0.857190,1.417976,-1.030847,-1.796829,1.351307,-0.181859,-0.978039,-1.544939,0.730446,1.790962,-1.202264,-0.739183,0.076715,-0.635999,-0.935414,4.519798
661,-1.231730,-0.218404,0.566850,0.975638,0.743827,0.494898,-2.441780,0.307874,0.122141,4.597391,-0.428228,-1.202264,-0.739183,0.076715,1.572330,-0.935414,-0.221249


Scaling complete. Train mean: -0.0000, Train std: 1.0003


## 10. Assembling the full preprocessing pipeline
Define a complete scikit-learn Pipeline incorporating outlier capping, feature engineering & selection, and scaling.


In [23]:
from sklearn .pipeline import Pipeline 
from src .features .outliers import IQROutlierCapper 
from src .features .preprocessing import FeatureEngineerSelector 
full_preprocessing_pipeline =Pipeline (steps =[
('outlier_capper',IQROutlierCapper (columns =NUM_CONT_COLS ,factor =1.5 )),
('feature_engineering',FeatureEngineerSelector (variance_threshold =0.02 ,corr_threshold =0.75 ,target_corr_threshold =0.05 )),
('scaler',StandardScaler ())
])
print ("Full Preprocessing Pipeline Architecture:")
print (full_preprocessing_pipeline )


Full Preprocessing Pipeline Architecture:
Pipeline(steps=[('outlier_capper',
                 IQROutlierCapper(columns=['Age', 'Sleep_Hours',
                                           'Study_Hours_Per_Week', 'GPA',
                                           'Social_Media_Hours',
                                           'Screen_Time_Hours',
                                           'Attendance_Rate'])),
                ('feature_engineering', FeatureEngineerSelector()),
                ('scaler', StandardScaler())])


## 11. Validating the pipeline
Perform sanity checks, leakage verification, and dimension consistency tests.


In [24]:
assert not x_train_scaled .isna ().any ().any (),"Found missing values in x_train_scaled!"
assert not x_test_scaled .isna ().any ().any (),"Found missing values in x_test_scaled!"
assert len (x_train_scaled )==len (y_train_encoded ),"Mismatch between x_train and y_train row counts!"
assert len (x_test_scaled )==len (y_test_encoded ),"Mismatch between x_test and y_test row counts!"
assert x_train_scaled .shape [1 ]==x_test_scaled .shape [1 ],"Feature count mismatch between train and test!"
print ("✓ All sanity checks passed successfully:")
print (f"  - No NaNs or inf values in train or test matrices.")
print (f"  - Feature shapes match: {x_train_scaled .shape [1 ]} features.")
print (f"  - Train rows: {len (x_train_scaled )}, Test rows: {len (x_test_scaled )}.")


✓ All sanity checks passed successfully:
  - No NaNs or inf values in train or test matrices.
  - Feature shapes match: 17 features.
  - Train rows: 1500, Test rows: 500.


### 11.5 Visual before-and-after comparison
Compare key feature distributions before and after preprocessing transformations.


In [25]:
fig ,axes =plt .subplots (2 ,2 ,figsize =(14 ,8 ))
sns .histplot (x_train ['Sleep_Hours'],kde =True ,ax =axes [0 ,0 ],color ='#3498db',bins =15 )
axes [0 ,0 ].set_title ('Sleep_Hours (Before Scaling / Raw Capped)')
sns .histplot (x_train ['Financial_Stress'],kde =True ,ax =axes [0 ,1 ],color ='#e67e22',bins =15 )
axes [0 ,1 ].set_title ('Financial_Stress (Before Scaling)')
sns .histplot (x_train_scaled ['Sleep_Hours'],kde =True ,ax =axes [1 ,0 ],color ='#2980b9',bins =15 )
axes [1 ,0 ].set_title ('Sleep_Hours (After StandardScaler: mean≈0, std≈1)')
sns .histplot (x_train_scaled ['Financial_Stress'],kde =True ,ax =axes [1 ,1 ],color ='#d35400',bins =15 )
axes [1 ,1 ].set_title ('Financial_Stress (After StandardScaler: mean≈0, std≈1)')
plt .tight_layout ()
plt .show ()


## 12. Saving the artefacts
Persist the prepared train/test splits and fitted preprocessing models for downstream training in notebook 03.


In [26]:
import joblib 
from src .data .load_data import save_data 
save_data ('train_features',x_train_scaled )
save_data ('test_features',x_test_scaled )
save_data ('train_target',pd .DataFrame ({'Mental_Health_Risk':y_train_encoded }))
save_data ('test_target',pd .DataFrame ({'Mental_Health_Risk':y_test_encoded }))
joblib .dump (scaler ,'models/scaler.joblib')
joblib .dump (full_preprocessing_pipeline ,'models/best_pipeline.joblib')
print ("✓ Successfully saved preprocessing artefacts:")
print ("  - data/processed/train_features.csv")
print ("  - data/processed/test_features.csv")
print ("  - data/processed/train_target.csv")
print ("  - data/processed/test_target.csv")
print ("  - models/scaler.joblib")
print ("  - models/best_pipeline.joblib")


✓ Successfully saved preprocessing artefacts:
  - data/processed/train_features.csv
  - data/processed/test_features.csv
  - data/processed/train_target.csv
  - data/processed/test_target.csv
  - models/scaler.joblib
  - models/best_pipeline.joblib


## 13. Summary

### 13.1 Decision log
Summary of preprocessing and transformation decisions implemented in this notebook.


In [27]:
decision_log =pd .DataFrame ([
{'Step':'1. Identifier Removal','Method':'Drop Student_ID','Justification':'Arbitrary identifier with zero predictive value.'},
{'Step':'2. Cleaning','Method':'Set Social_Media > Screen_Time to NaN; text capitalization','Justification':'Fixed domain contradictions and inconsistent text.'},
{'Step':'3. Train/Test Split','Method':'Stratified 80/20 Split (seed=42)','Justification':'Preserves class balance while isolating test set.'},
{'Step':'4. Missing Values','Method':'Median (num) & Most Frequent (cat) imputation','Justification':'Robust imputation without leaking test information.'},
{'Step':'5. Outlier Strategy','Method':'1.5 * IQR Capping (Winsorization) on train bounds','Justification':'Clips extreme anomalies without discarding rows.'},
{'Step':'6. Feature Engineering','Method':'Domain candidates + Variance & Correlation filters','Justification':'Retains high-impact non-redundant features (Effort Index, Stress/Support ratio).'},
{'Step':'7. Encoding','Method':'OrdinalEncoder + OneHotEncoder(drop="first") + Binary 0/1','Justification':'Preserves natural order and prevents multicollinearity.'},
{'Step':'8. Class Imbalance','Method':'No Resampling','Justification':'Classes are naturally balanced (~33% each).'},
{'Step':'9. Scaling','Method':'StandardScaler','Justification':'Normalizes features for distance- and gradient-based models.'}
])
display (decision_log )
print (f"\nFinal processed dataset dimensions:")
print (f"  - Train: {x_train_scaled .shape [0 ]} rows x {x_train_scaled .shape [1 ]} features")
print (f"  - Test:  {x_test_scaled .shape [0 ]} rows x {x_test_scaled .shape [1 ]} features")
print ("Ready for model training in notebook 03.")


,Step,Method,Justification
0,1. Identifier Removal,Drop Student_ID,Arbitrary identifier with zero predictive value.
1,2. Cleaning,Set Social_Media > Screen_Time to NaN; text capitalization,Fixed domain contradictions and inconsistent text.
2,3. Train/Test Split,Stratified 80/20 Split (seed=42),Preserves class balance while isolating test set.
3,4. Missing Values,Median (num) & Most Frequent (cat) imputation,Robust imputation without leaking test information.
4,5. Outlier Strategy,1.5 * IQR Capping (Winsorization) on train bounds,Clips extreme anomalies without discarding rows.
5,6. Feature Engineering,Domain candidates + Variance & Correlation filters,"Retains high-impact non-redundant features (Effort Index, Stress/Support ratio)."
6,7. Encoding,"OrdinalEncoder + OneHotEncoder(drop=""first"") + Binary 0/1",Preserves natural order and prevents multicollinearity.
7,8. Class Imbalance,No Resampling,Classes are naturally balanced (~33% each).
8,9. Scaling,StandardScaler,Normalizes features for distance- and gradient-based models.



Final processed dataset dimensions:
  - Train: 1500 rows x 17 features
  - Test:  500 rows x 17 features
Ready for model training in notebook 03.
